In [1]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [2]:
%%capture
from protest_impact.data.protests.detection import load_aglpn_dataset, load_glpn_dataset

glpn = load_glpn_dataset()
aglpn = load_aglpn_dataset()

In [3]:
import optuna
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from datasets import concatenate_datasets, DatasetDict
from protest_impact.data.news import kwic_dataset
from protest_impact.data.protests.detection import load_aglpn_dataset, load_glpn_dataset
from protest_impact.data.protests.detection.train import evaluate_, train_model


def objective(trial, return_model=False):
    dataset_name = trial.suggest_categorical(
        "dataset",
        [
            "aglpn",
            "aglpn+glpn",
            "aglpn+aglpn.positive",
            "aglpn+aglpn.positive+glpn",
        ],
    )
    if dataset_name == "aglpn":
        train = aglpn["train"]
    elif dataset_name == "aglpn+glpn":
        train = concatenate_datasets(
            [aglpn["train"], glpn["train"]]
        )
    elif dataset_name == "aglpn+aglpn.positive":
        train = concatenate_datasets(
            [aglpn["train"], aglpn["train.positive"]]
        )
    elif dataset_name == "aglpn+aglpn.positive+glpn":
        train = concatenate_datasets(
            [
                aglpn["train"],
                aglpn["train.positive"],
                glpn["train"],
            ]
        )
    train = train.shuffle(seed=20230212)
    n_kwic = trial.suggest_int("n_kwic", 1, 5)
    train = kwic_dataset(train, n=n_kwic)
    test = load_aglpn_dataset()["test"]
    test = kwic_dataset(test, n=n_kwic)
    dataset = DatasetDict({"train": train, "test": test})
    model_name = "deepset/gelectra-large"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_vanilla = AutoModelForSequenceClassification.from_pretrained(model_name).to(
        device
    )
    n_epochs = trial.suggest_int("n_epochs", 2, 2)
    model = train_model(model_vanilla, tokenizer, dataset_name, dataset, n_epochs=n_epochs)
    if return_model:
        return model
    evaluation = evaluate_(model, tokenizer, test)
    return evaluation["f1"]


study = optuna.create_study(
    direction="maximize",
    study_name="protest_detection_v0",
    load_if_exists=True,
    storage="sqlite:///db.sqlite3",
)
# study.optimize(objective, n_trials=5)

[I 2023-02-12 14:45:28,754] Using an existing study with name 'protest_detection_v0' instead of creating a new one.


In [4]:
study.best_params

{'dataset': 'aglpn+aglpn.positive', 'n_epochs': 2, 'n_kwic': 4}

In [7]:
from optuna.visualization import *

# plot_parallel_coordinate(study)
plot_optimization_history(study)